# LeetCode 3149 — Find the Minimum Cost Array Permutation bằng Python

- Link: https://leetcode.com/problems/find-the-minimum-cost-array-permutation/
- Độ khó: **Hard**
- Chủ đề: **DP bitmask kiểu TSP (Held–Karp)** trên chu trình; điểm số **bất biến khi xoay vòng** ⇒ cố định `perm[0] = 0`; dựng hoán vị **nhỏ nhất theo thứ tự từ điển** bằng DP "chi phí còn lại" rồi chọn tham lam

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findPermutation(self, nums: List[int]) -> List[int]:
```

## 1. Mục tiêu học

1. Đọc điểm số như **độ dài chu trình**: cạnh `perm[i] → perm[i+1]` có trọng số `|perm[i] − nums[perm[i+1]]|`, kể cả cạnh đóng vòng `perm[n−1] → perm[0]`.
2. Xoay vòng hoán vị không đổi điểm ⇒ luôn có lời giải tối ưu bắt đầu bằng `0`, và lời giải nhỏ nhất theo từ điển **phải** bắt đầu bằng `0`.
3. DP `f[mask][last]` = chi phí nhỏ nhất để **đi tiếp** từ `last` qua các đỉnh chưa thăm rồi về `0` — tính ngược.
4. Dựng kết quả: từ `0`, mỗi bước chọn đỉnh **nhỏ nhất** giữ được chi phí tối ưu ⇒ nhỏ nhất theo từ điển.
5. Kiểm chứng bằng duyệt mọi hoán vị (`n ≤ 8`).

## 2. Đề bài & ví dụ

`nums` là một hoán vị của `[0, …, n−1]`. Với hoán vị `perm`, điểm là

`score(perm) = |perm[0] − nums[perm[1]]| + |perm[1] − nums[perm[2]]| + … + |perm[n−1] − nums[perm[0]]|`.

Trả về `perm` có điểm nhỏ nhất; nếu nhiều, chọn cái nhỏ nhất theo thứ tự từ điển.

**Ví dụ 1:** `nums = [1,0,2]` → `[0,1,2]` (điểm 2). **Ví dụ 2:** `nums = [0,2,1]` → `[0,2,1]` (điểm 2).

**Ràng buộc:** `2 ≤ n ≤ 14`.

In [1]:
from typing import List
from functools import lru_cache
import itertools, random, time

examples = [([1, 0, 2], [0, 1, 2]), ([0, 2, 1], [0, 2, 1])]
def score(perm, nums):
    n = len(perm)
    return sum(abs(perm[i] - nums[perm[(i + 1) % n]]) for i in range(n))

for nums, exp in examples:
    print(nums, '-> kỳ vọng', exp, '| điểm', score(exp, nums))

[1, 0, 2] -> kỳ vọng [0, 1, 2] | điểm 2
[0, 2, 1] -> kỳ vọng [0, 2, 1] | điểm 2


## 3. Kiến thức nền

### 3.1. Điểm số bất biến khi xoay

Mỗi số hạng dùng cặp **liên tiếp trên vòng** `(perm[i], perm[i+1 mod n])`. Xoay `perm` chỉ đổi chỉ số bắt đầu ⇒ tập cặp không đổi ⇒ điểm không đổi.

In [2]:
p = [2, 0, 3, 1]; nums = [3, 1, 0, 2]
for r in range(4):
    q = p[r:] + p[:r]
    print(q, '-> điểm', score(q, nums))

[2, 0, 3, 1] -> điểm 6
[0, 3, 1, 2] -> điểm 6
[3, 1, 2, 0] -> điểm 6
[1, 2, 0, 3] -> điểm 6


Hệ quả: trong mọi lớp xoay của một hoán vị tối ưu có một hoán vị bắt đầu bằng `0`, và nó nhỏ hơn theo từ điển mọi hoán vị bắt đầu bằng số khác ⇒ **đáp án bắt đầu bằng 0**.

### 3.2. Held–Karp (TSP bằng bitmask)

`f[mask][u]`: đã thăm tập `mask` (gồm `0` và `u`), đang ở `u`; giá trị = chi phí nhỏ nhất của phần đường **còn lại** (thăm hết đỉnh chưa thăm rồi đóng vòng về `0`):

- `f[full][u] = |u − nums[0]|`;
- `f[mask][u] = min_{v ∉ mask} |u − nums[v]| + f[mask | v][v]`.

Đáp án tối ưu là `f[{0}][0]`. Số trạng thái `2^n · n ≈ 2.3·10⁵`, mỗi trạng thái `O(n)` ⇒ `≈ 3·10⁶`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Vì `f` là chi phí **phía trước**, khi đứng ở `u` với tập `mask` ta biết ngay đi tới `v` nào thì vẫn đạt tối ưu: những `v` có `|u − nums[v]| + f[mask|v][v] = f[mask][u]`. Chọn `v` nhỏ nhất trong số đó — các lựa chọn sau không ảnh hưởng tới vị trí đã chốt ⇒ kết quả nhỏ nhất theo từ điển. (Nếu tính DP "chi phí đã đi" thì truy vết ngược cho hoán vị tối ưu nhưng **không** dễ đảm bảo thứ tự từ điển.)

## 5. Cách 1 — Brute Force: thử mọi hoán vị

In [3]:
def brute(nums):
    n = len(nums); best = None
    for p in itertools.permutations(range(n)):          # sinh theo thứ tự từ điển
        s = score(p, nums)
        if best is None or s < best[0]:
            best = (s, list(p))
    return best[1]

for nums, exp in examples:
    print('brute:', brute(nums), '| kỳ vọng', exp)

brute: [0, 1, 2] | kỳ vọng [0, 1, 2]
brute: [0, 2, 1] | kỳ vọng [0, 2, 1]


`itertools.permutations(range(n))` sinh theo thứ tự từ điển nên giữ cái **đầu tiên** đạt min là đủ. `O(n! · n)` — chỉ cho `n ≤ 8`.

### 5.1. Brute có cố định `perm[0] = 0`

Chỉ còn `(n − 1)!` hoán vị — vẫn đúng theo mục 3.1.

In [4]:
def brute_fixed0(nums):
    n = len(nums); best = None
    for rest in itertools.permutations(range(1, n)):
        p = (0,) + rest; s = score(p, nums)
        if best is None or s < best[0]:
            best = (s, list(p))
    return best[1]

for nums, exp in examples:
    print('brute cố định 0:', brute_fixed0(nums), '| kỳ vọng', exp)

brute cố định 0: [0, 1, 2] | kỳ vọng [0, 1, 2]
brute cố định 0: [0, 2, 1] | kỳ vọng [0, 2, 1]


## 6. Ý tưởng thuật toán / bước đột phá

```
f[full][u] = |u − nums[0]|
for mask giảm dần (chứa bit 0):
    for u ∈ mask: f[mask][u] = min_{v ∉ mask} |u − nums[v]| + f[mask|v][v]
perm = [0]; mask = 1; u = 0
lặp n − 1 lần: chọn v nhỏ nhất ∉ mask với |u − nums[v]| + f[mask|v][v] == f[mask][u]
```

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP lặp chi phí phía trước + dựng tham lam (lời giải chính)

In [5]:
def held_karp_forward(nums):
    n = len(nums); full = (1 << n) - 1; INF = float('inf')
    f = [[INF] * n for _ in range(1 << n)]
    for u in range(n):
        f[full][u] = abs(u - nums[0])
    for mask in range(full - 1, 0, -1):
        if not mask & 1:
            continue
        row = f[mask]
        for u in range(n):
            if not mask >> u & 1:
                continue
            best = INF
            for v in range(n):
                if not mask >> v & 1:
                    c = abs(u - nums[v]) + f[mask | 1 << v][v]
                    if c < best:
                        best = c
            row[u] = best
    perm = [0]; mask = 1; u = 0
    for _ in range(n - 1):
        for v in range(n):
            if not mask >> v & 1 and abs(u - nums[v]) + f[mask | 1 << v][v] == f[mask][u]:
                perm.append(v); mask |= 1 << v; u = v
                break
    return perm

for nums, exp in examples:
    print('Held-Karp:', held_karp_forward(nums), '| kỳ vọng', exp)

Held-Karp: [0, 1, 2] | kỳ vọng [0, 1, 2]
Held-Karp: [0, 2, 1] | kỳ vọng [0, 2, 1]


### 7.2. Cách 3 — DFS có nhớ (`lru_cache`) cùng ý tưởng

In [6]:
def memo_version(nums):
    n = len(nums); full = (1 << n) - 1
    @lru_cache(maxsize=None)
    def f(mask, u):
        if mask == full:
            return abs(u - nums[0])
        return min(abs(u - nums[v]) + f(mask | 1 << v, v) for v in range(n) if not mask >> v & 1)
    perm = [0]; mask = 1; u = 0
    while mask != full:
        target = f(mask, u)
        v = next(v for v in range(n) if not mask >> v & 1 and abs(u - nums[v]) + f(mask | 1 << v, v) == target)
        perm.append(v); mask |= 1 << v; u = v
    return perm

for nums, exp in examples:
    print('DFS có nhớ:', memo_version(nums), '| kỳ vọng', exp)

DFS có nhớ: [0, 1, 2] | kỳ vọng [0, 1, 2]
DFS có nhớ: [0, 2, 1] | kỳ vọng [0, 2, 1]


### 7.3. Cách 4 — DP chi phí đã đi + so sánh **cả dãy** (từ điển) trong trạng thái

`g[mask][u]` = cặp `(chi phí đã đi từ 0 tới u qua mask, dãy đường đi)` nhỏ nhất theo thứ tự `(chi phí, dãy)`. Đúng nhưng tốn bộ nhớ/thời gian vì lưu cả dãy — để minh hoạ vì sao "chi phí phía trước" gọn hơn. **Chú ý:** cuối cùng ta so `(tổng chi phí, dãy)` trên các trạng thái đầy; ở trạng thái trung gian, giữ cặp `(chi phí, dãy)` nhỏ nhất là an toàn vì phần đuôi chỉ phụ thuộc `(mask, u)` — kiểm chứng bằng brute ở mục 12.

In [7]:
def dp_with_paths(nums):
    n = len(nums); full = (1 << n) - 1
    g = {(1, 0): (0, (0,))}
    for mask in range(1, full + 1):
        if not mask & 1:
            continue
        for u in range(n):
            st = g.get((mask, u))
            if st is None:
                continue
            c, path = st
            for v in range(n):
                if not mask >> v & 1:
                    key = (mask | 1 << v, v); cand = (c + abs(u - nums[v]), path + (v,))
                    if key not in g or cand < g[key]:
                        g[key] = cand
    best = min((c + abs(u - nums[0]), path) for (m, u), (c, path) in g.items() if m == full)
    return list(best[1])

for nums, exp in examples:
    print('DP lưu dãy:', dp_with_paths(nums), '| kỳ vọng', exp)

DP lưu dãy: [0, 1, 2] | kỳ vọng [0, 1, 2]
DP lưu dãy: [0, 2, 1] | kỳ vọng [0, 2, 1]


## 8. TRACE từng bước

Trace bước dựng của cách 2 trên `nums = [3, 1, 0, 4, 2]`: ở mỗi bước in các ứng viên và chi phí `|u − nums[v]| + f[mask|v][v]`.

In [8]:
def trace(nums):
    n = len(nums); full = (1 << n) - 1
    @lru_cache(maxsize=None)
    def f(mask, u):
        if mask == full:
            return abs(u - nums[0])
        return min(abs(u - nums[v]) + f(mask | 1 << v, v) for v in range(n) if not mask >> v & 1)
    perm = [0]; mask = 1; u = 0
    print('chi phí tối ưu f({0}, 0) =', f(1, 0))
    while mask != full:
        cands = [(v, abs(u - nums[v]) + f(mask | 1 << v, v)) for v in range(n) if not mask >> v & 1]
        v = min(v for v, c in cands if c == f(mask, u))
        print(f'đang ở {u}, đã thăm {bin(mask)}: ứng viên (v, chi phí) = {cands} -> chọn {v}')
        perm.append(v); mask |= 1 << v; u = v
    print('perm =', perm, '| điểm', score(perm, nums), '| brute:', brute(nums))

trace([3, 1, 0, 4, 2])

chi phí tối ưu f({0}, 0) = 2
đang ở 0, đã thăm 0b1: ứng viên (v, chi phí) = [(1, 2), (2, 2), (3, 8), (4, 6)] -> chọn 1
đang ở 1, đã thăm 0b11: ứng viên (v, chi phí) = [(2, 1), (3, 7), (4, 5)] -> chọn 2
đang ở 2, đã thăm 0b111: ứng viên (v, chi phí) = [(3, 4), (4, 0)] -> chọn 4
đang ở 4, đã thăm 0b10111: ứng viên (v, chi phí) = [(3, 0)] -> chọn 3
perm = [0, 1, 2, 4, 3] | điểm 2 | brute: [0, 1, 2, 4, 3]


## 9. Code nộp LeetCode

Chỉ duyệt các `mask` **lẻ** (chứa đỉnh 0): bắt đầu từ `full − 2` (lẻ lớn nhất khác `full`), bước `−2`.

In [9]:
class Solution:
    def findPermutation(self, nums: List[int]) -> List[int]:
        n = len(nums)
        full = (1 << n) - 1
        INF = float('inf')
        f = [[INF] * n for _ in range(1 << n)]
        for u in range(n):
            f[full][u] = abs(u - nums[0])
        cost = [[abs(u - nums[v]) for v in range(n)] for u in range(n)]
        for mask in range(full - 2, 0, -2):          # mask lẻ: chứa đỉnh 0
            row = f[mask]
            free = [v for v in range(n) if not mask >> v & 1]
            for u in range(n):
                if mask >> u & 1:
                    cu = cost[u]
                    best = INF
                    for v in free:
                        c = cu[v] + f[mask | (1 << v)][v]
                        if c < best:
                            best = c
                    row[u] = best
        perm = [0]
        mask = 1
        u = 0
        for _ in range(n - 1):
            for v in range(n):
                if not mask >> v & 1 and cost[u][v] + f[mask | (1 << v)][v] == f[mask][u]:
                    perm.append(v)
                    mask |= 1 << v
                    u = v
                    break
        return perm

for nums, exp in examples:
    got = Solution().findPermutation(nums)
    print(got, 'OK' if got == exp else 'SAI')

[0, 1, 2] OK
[0, 2, 1] OK


## 10. Vì sao thuật toán đúng

1. **Bắt đầu bằng 0:** mục 3.1 — tồn tại tối ưu bắt đầu bằng 0, và mọi hoán vị bắt đầu bằng 0 nhỏ hơn (từ điển) mọi hoán vị bắt đầu bằng số khác.
2. **DP:** `f[mask][u]` đúng định nghĩa "chi phí phía trước nhỏ nhất" bằng quy nạp giảm dần theo `mask` (mọi `mask | v` lớn hơn `mask` đã tính trước).
3. **Tham lam từ điển:** giả sử đã chốt tiền tố `perm[0..i]` (tập `mask`, đỉnh cuối `u`) là tiền tố của một tối ưu. Các `v` có `cost(u, v) + f[mask|v][v] = f[mask][u]` là đúng các lựa chọn còn mở rộng được thành tối ưu. Chọn `v` nhỏ nhất làm vị trí `i + 1` nhỏ nhất có thể ⇒ theo quy nạp ra hoán vị tối ưu nhỏ nhất theo từ điển.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(n! · n)` | `O(n)` |
| 2. Held–Karp lặp | `O(2^n · n²)` | `O(2^n · n)` |
| 3. DFS có nhớ | `O(2^n · n²)` | `O(2^n · n)` |
| 4. DP lưu dãy | `O(2^n · n² · n)` | `O(2^n · n · n)` |

`n = 14`: `2^{14} · 14² ≈ 3.2·10⁶` — khoảng 1–2 s trong Python (chỉ mask lẻ: một nửa).

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'forward': held_karp_forward, 'memo': memo_version, 'paths': dp_with_paths,
         'submit': lambda a: Solution().findPermutation(a)}
edge = [([0, 1], [0, 1]), ([1, 0], [0, 1]), ([2, 0, 1], [0, 1, 2]), ([0, 1, 2, 3], [0, 1, 2, 3]), ([3, 2, 1, 0], [0, 2, 1, 3])]
for a, exp in examples + edge:
    assert brute(a) == exp, (a, brute(a))
    for name, f in impls.items():
        assert f(a) == exp, (name, a)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với brute (`n ≤ 8`)

In [11]:
rng = random.Random(3149)
cnt = 0
for _ in range(200):
    n = rng.randint(2, 8); a = rng.sample(range(n), n)
    exp = brute(a) if n <= 7 else brute_fixed0(a)
    assert brute_fixed0(a) == exp
    for name, f in impls.items():
        assert f(a) == exp, (name, a)
    cnt += 1
print('Khớp brute trên', cnt, 'hoán vị ngẫu nhiên.')

Khớp brute trên 200 hoán vị ngẫu nhiên.


### 12.3. Ca lớn `n = 14`

In [12]:
cases = {'ngẫu nhiên': rng.sample(range(14), 14), 'đồng nhất': list(range(14)), 'đảo ngược': list(range(13, -1, -1))}
for name, a in cases.items():
    t = time.time(); p = Solution().findPermutation(a); t1 = time.time() - t
    q = memo_version(a)
    assert p == q and sorted(p) == list(range(14))
    print(f'{name:10s}: {p} | điểm {score(p, a)} | nộp {t1:.2f}s')

ngẫu nhiên: [0, 3, 13, 1, 5, 10, 2, 7, 4, 9, 12, 11, 8, 6] | điểm 2 | nộp 0.04s


đồng nhất : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13] | điểm 26 | nộp 0.03s


đảo ngược : [0, 7, 6, 8, 5, 9, 4, 10, 3, 11, 2, 12, 1, 13] | điểm 12 | nộp 0.04s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên cạnh đóng vòng `|perm[n−1] − nums[perm[0]]|`

In [13]:
def wrong_no_cycle(nums):
    n = len(nums); best = None
    for p in itertools.permutations(range(n)):
        s = sum(abs(p[i] - nums[p[i + 1]]) for i in range(n - 1))         # SAI: thiếu cạnh cuối
        if best is None or s < best[0]:
            best = (s, list(p))
    return best[1]

a = next(a for a in (rng.sample(range(5), 5) for _ in range(500)) if wrong_no_cycle(a) != Solution().findPermutation(a))
print('nums =', a, '| sai :', wrong_no_cycle(a), score(wrong_no_cycle(a), a), '| đúng:', Solution().findPermutation(a), score(Solution().findPermutation(a), a))

nums = [4, 1, 0, 2, 3] | sai : [1, 2, 3, 4, 0] 2 | đúng: [0, 1, 2, 3, 4] 2


**Vì sao sai:** đường đi không đóng có thể rẻ hơn nhưng khi đóng vòng lại đắt; còn mất luôn tính bất biến khi xoay (không còn chắc bắt đầu bằng 0).

### Lỗi 2 — Dựng tham lam "đỉnh gần nhất" (không dùng DP)

In [14]:
def wrong_nearest(nums):
    n = len(nums); perm = [0]; used = {0}; u = 0
    for _ in range(n - 1):
        v = min((v for v in range(n) if v not in used), key=lambda v: (abs(u - nums[v]), v))   # SAI
        perm.append(v); used.add(v); u = v
    return perm

diff = next(a for a in (rng.sample(range(6), 6) for _ in range(500)) if wrong_nearest(a) != Solution().findPermutation(a))
print('nums =', diff, '| sai :', wrong_nearest(diff), score(wrong_nearest(diff), diff),
      '| đúng:', Solution().findPermutation(diff), score(Solution().findPermutation(diff), diff))

nums = [4, 3, 1, 2, 5, 0] | sai : [0, 5, 4, 1, 2, 3] 2 | đúng: [0, 2, 3, 1, 5, 4] 2


**Vì sao sai:** tham lam láng giềng gần nhất cho TSP không tối ưu — bước rẻ bây giờ có thể buộc các bước sau (và cạnh đóng vòng) rất đắt. Ở phản ví dụ tìm được, hai hoán vị cùng điểm nhưng tham lam trả về hoán vị **lớn hơn theo từ điển** (vì ở bước đầu chọn đỉnh có cạnh rẻ nhất thay vì đỉnh nhỏ nhất vẫn giữ tối ưu); trên các bộ khác, nó còn cho điểm lớn hơn hẳn.

### Lỗi 3 — Đảo chiều trọng số: `|nums[perm[i]] − perm[i+1]|`

In [15]:
def wrong_direction(nums):
    n = len(nums); best = None
    for p in itertools.permutations(range(n)):
        s = sum(abs(nums[p[i]] - p[(i + 1) % n]) for i in range(n))       # SAI
        if best is None or s < best[0]:
            best = (s, list(p))
    return best[1]

a = examples[0][0]
print('sai :', wrong_direction(a), '| đúng:', Solution().findPermutation(a))

sai : [0, 2, 1] | đúng: [0, 1, 2]


**Vì sao sai:** cạnh `perm[i] → perm[i+1]` so `perm[i]` với `nums` của **phần tử sau**, không phải `nums` của chính nó. Ví dụ 1 cho `[0, 2, 1]` thay vì `[0, 1, 2]`.

### Lỗi 4 — Bước nhảy `−2` từ `full − 1` (duyệt mask chẵn)

In [16]:
def wrong_mask_loop(nums):
    n = len(nums); full = (1 << n) - 1; INF = float('inf')
    f = [[INF] * n for _ in range(1 << n)]
    for u in range(n):
        f[full][u] = abs(u - nums[0])
    for mask in range(full - 1, 0, -2):                                   # SAI: mask chẵn
        for u in range(n):
            if mask >> u & 1:
                f[mask][u] = min((abs(u - nums[v]) + f[mask | 1 << v][v] for v in range(n) if not mask >> v & 1), default=INF)
    return f[1][0]

a = [0, 1, 2, 3]
print('chi phí tối ưu (sai):', wrong_mask_loop(a), '| đúng:', score(Solution().findPermutation(a), a))

chi phí tối ưu (sai): inf | đúng: 6


**Vì sao sai:** `full − 1` chẵn ⇒ dãy `full − 1, full − 3, …` toàn số chẵn — không trạng thái nào chứa đỉnh 0 được tính, `f[1][0] = ∞`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `n ≤ 14–16` + hoán vị + chi phí cặp liên tiếp ⇒ DP bitmask (TSP).
- Chu trình ⇒ cố định đỉnh đầu.
- Yêu cầu **nhỏ nhất theo từ điển** ⇒ DP "chi phí phía trước" + chọn tham lam từ đầu.

## 15. Biến thể của bài

1. Đường đi (không đóng vòng) ⇒ không cố định đỉnh đầu được; thử mọi đỉnh đầu, `f[full][u] = 0`.
2. Hoán vị **lớn nhất** theo từ điển trong các tối ưu ⇒ chọn `v` lớn nhất ở mỗi bước — nhưng đỉnh đầu không còn chắc là 0 (xoay vòng: phải bắt đầu bằng `n − 1`).
3. Điểm **lớn nhất** ⇒ đổi `min` thành `max`.
4. Đếm số hoán vị tối ưu ⇒ DP đếm song song với DP chi phí.

In [17]:
# Biến thể 4: đếm số hoán vị đạt điểm nhỏ nhất
def count_optimal(nums):
    n = len(nums); full = (1 << n) - 1
    @lru_cache(maxsize=None)
    def f(mask, u):                       # (chi phí min phía trước, số cách) — cố định perm[0] = 0
        if mask == full:
            return abs(u - nums[0]), 1
        best, ways = float('inf'), 0
        for v in range(n):
            if not mask >> v & 1:
                c, w = f(mask | 1 << v, v); c += abs(u - nums[v])
                if c < best: best, ways = c, w
                elif c == best: ways += w
        return best, ways
    c, w = f(1, 0)
    return w * n                          # mỗi chu trình có n cách xoay (n phép xoay khác nhau)

def count_brute(nums):
    n = len(nums); sc = [score(p, nums) for p in itertools.permutations(range(n))]
    return sc.count(min(sc))

for _ in range(100):
    n = rng.randint(2, 7); a = rng.sample(range(n), n)
    assert count_optimal(a) == count_brute(a)
print('Biến thể đếm hoán vị tối ưu: khớp brute (= số đường từ 0 × n phép xoay).')

Biến thể đếm hoán vị tối ưu: khớp brute (= số đường từ 0 × n phép xoay).


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi hoán vị | `O(n!·n)` | Oracle |
| 2 | Held–Karp chi phí phía trước | `O(2^n n²)` | **Nộp bài** |
| 3 | DFS có nhớ | `O(2^n n²)` | Ngắn |
| 4 | DP lưu cả dãy | `O(2^n n³)` | Minh hoạ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [0, 1]`.

**Bài 2 (tính tay):** `nums = [1, 2, 0]` — tính điểm của `[0, 1, 2]` và `[0, 2, 1]`.

**Bài 3 (điền TODO):** biến thể 1 — đường đi không đóng vòng, trả hoán vị tối ưu nhỏ nhất theo từ điển.

**Bài 4 (tự viết lại):** biến thể 3 — điểm **lớn nhất**, nhỏ nhất theo từ điển.

**Bài 5 (mini challenge):** trả về hoán vị tối ưu **thứ `k`** theo thứ tự từ điển (1-index), hoặc `None`.

In [18]:
def path_version_todo(nums):
    # TODO
    raise NotImplementedError

def max_score_todo(nums):
    # TODO
    raise NotImplementedError

def kth_optimal_todo(nums, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Hai hoán vị `[0, 1]`, `[1, 0]` là xoay của nhau. (2) Cùng điểm. (3) `[0, 1]`.

**Bài 2.** (1) `[0,1,2]`: `|0 − 2| + |1 − 0| + |2 − 1| = 4`. (2) `[0,2,1]`: `|0 − 0| + |2 − 2| + |1 − 1| = 0`. (3) `[0, 2, 1]`.

**Bài 3.** (1) Không cố định đỉnh đầu; `f[full][u] = 0`. (2) Thử đỉnh đầu từ nhỏ tới lớn, lấy đỉnh đầu nhỏ nhất có tối ưu toàn cục. (3) Dựng tham lam như cũ.

**Bài 4.** (1) Đổi `min` thành `max`. (2) Vẫn cố định 0 (xoay không đổi điểm). (3) Tham lam chọn `v` nhỏ nhất giữ được `max`.

**Bài 5.** (1) Đếm số cách hoàn thành tối ưu `ways[mask][u]`. (2) Ở mỗi bước duyệt `v` tăng dần; nếu `k > ways` của nhánh thì trừ và bỏ qua. (3) Kiểm bằng liệt kê.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 4
assert Solution().findPermutation([0, 1]) == [0, 1]
assert Solution().findPermutation([1, 2, 0]) == [0, 2, 1] and score([0, 2, 1], [1, 2, 0]) == 0

def max_score(nums):
    n = len(nums); full = (1 << n) - 1
    @lru_cache(maxsize=None)
    def f(mask, u):
        if mask == full:
            return abs(u - nums[0])
        return max(abs(u - nums[v]) + f(mask | 1 << v, v) for v in range(n) if not mask >> v & 1)
    perm = [0]; mask = 1; u = 0
    while mask != full:
        v = next(v for v in range(n) if not mask >> v & 1 and abs(u - nums[v]) + f(mask | 1 << v, v) == f(mask, u))
        perm.append(v); mask |= 1 << v; u = v
    return perm

def max_brute(nums):
    best = None
    for p in itertools.permutations(range(len(nums))):
        s = score(p, nums)
        if best is None or s > best[0]:
            best = (s, list(p))
    return best[1]

for _ in range(100):
    n = rng.randint(2, 7); a = rng.sample(range(n), n)
    assert max_score(a) == max_brute(a)
print('Bài 1: [0, 1] | Bài 2: [0, 2, 1] (điểm 0) | Bài 4: điểm lớn nhất khớp brute.')

Bài 1: [0, 1] | Bài 2: [0, 2, 1] (điểm 0) | Bài 4: điểm lớn nhất khớp brute.


In [20]:
# Bài 3, 5
def path_version(nums):
    n = len(nums); full = (1 << n) - 1
    @lru_cache(maxsize=None)
    def f(mask, u):
        if mask == full:
            return 0
        return min(abs(u - nums[v]) + f(mask | 1 << v, v) for v in range(n) if not mask >> v & 1)
    opt = min(f(1 << s, s) for s in range(n))
    s = min(s for s in range(n) if f(1 << s, s) == opt)
    perm = [s]; mask = 1 << s; u = s
    while mask != full:
        v = next(v for v in range(n) if not mask >> v & 1 and abs(u - nums[v]) + f(mask | 1 << v, v) == f(mask, u))
        perm.append(v); mask |= 1 << v; u = v
    return perm

def path_brute(nums):
    n = len(nums); best = None
    for p in itertools.permutations(range(n)):
        s = sum(abs(p[i] - nums[p[i + 1]]) for i in range(n - 1))
        if best is None or s < best[0]:
            best = (s, list(p))
    return best[1]

def kth_optimal(nums, k):
    n = len(nums); full = (1 << n) - 1
    @lru_cache(maxsize=None)
    def f(mask, u):
        if mask == full:
            return abs(u - nums[0]), 1
        best, ways = float('inf'), 0
        for v in range(n):
            if not mask >> v & 1:
                c, w = f(mask | 1 << v, v); c += abs(u - nums[v])
                if c < best: best, ways = c, w
                elif c == best: ways += w
        return best, ways
    # hoán vị tối ưu theo thứ tự từ điển: có thể bắt đầu bằng bất kỳ đỉnh nào (xoay) — liệt kê theo đỉnh đầu
    opt = f(1, 0)[0]
    def walk(start, k):
        # đếm đường bắt đầu ở start: dùng DP tương tự với đỉnh gốc là start
        @lru_cache(maxsize=None)
        def g(mask, u):
            if mask == full:
                return abs(u - nums[start]), 1
            best, ways = float('inf'), 0
            for v in range(n):
                if not mask >> v & 1:
                    c, w = g(mask | 1 << v, v); c += abs(u - nums[v])
                    if c < best: best, ways = c, w
                    elif c == best: ways += w
            return best, ways
        if g(1 << start, start)[1] < k:
            return None, k - g(1 << start, start)[1]
        perm = [start]; mask = 1 << start; u = start; spent = 0
        while mask != full:
            for v in range(n):
                if mask >> v & 1:
                    continue
                c, w = g(mask | 1 << v, v)
                if spent + abs(u - nums[v]) + c != opt:
                    continue
                if k > w:
                    k -= w; continue
                spent += abs(u - nums[v]); perm.append(v); mask |= 1 << v; u = v
                break
        return perm, 0
    for start in range(n):
        p, k = walk(start, k)
        if p is not None:
            return p
    return None

for _ in range(100):
    n = rng.randint(2, 6); a = rng.sample(range(n), n)
    assert path_version(a) == path_brute(a)
    opts = [list(p) for p in itertools.permutations(range(n)) if score(p, a) == min(score(q, a) for q in itertools.permutations(range(n)))]
    for k in range(1, len(opts) + 2):
        assert kth_optimal(a, k) == (opts[k - 1] if k <= len(opts) else None)
print('Bài 3 (đường đi) và Bài 5 (hoán vị tối ưu thứ k) khớp brute.')

Bài 3 (đường đi) và Bài 5 (hoán vị tối ưu thứ k) khớp brute.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
f[full][u] = |u − nums[0]|
for mask in range(full − 2, 0, −2):          # mask lẻ
    for u in mask: f[mask][u] = min_{v ∉ mask} |u − nums[v]| + f[mask|v][v]
perm = [0]; chọn tham lam v nhỏ nhất giữ đẳng thức f
```

**Checklist:**

- [ ] Có cạnh đóng vòng?
- [ ] Trọng số `|perm[i] − nums[perm[i+1]]|` đúng chiều?
- [ ] Cố định `perm[0] = 0`?
- [ ] DP chi phí **phía trước** để dựng từ điển nhỏ nhất?
- [ ] Vòng lặp mask duyệt đúng các mask chứa 0?